In [8]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import math
from pathlib import Path
import matplotlib.pyplot as plt

In [9]:
InputDir = "TestDataset"
OutputDir = "TestDatasetRes"
ModelPath = "PrismModel.pth"
BucketsPath = "CoordBuckets.npy"
Temperature = 0.38

NumHeads = 8
HeadDims = 32
EmbedDim = 256

In [10]:
class Attention(nn.Module):
	def __init__(self):
		super().__init__()

		self.QueryMatrix = nn.Linear(EmbedDim,EmbedDim)
		self.KeyMatrix = nn.Linear(EmbedDim,EmbedDim)
		self.ValueMatrix = nn.Linear(EmbedDim,EmbedDim)
		self.OutputMatrix = nn.Linear(EmbedDim,EmbedDim)
		self.Softmax = nn.Softmax(dim=-1)

	def forward(self,x):
		# SeqLen = 256 currently
		BatchSize,SeqLen,_ = x.shape

		QueryOutput = self.QueryMatrix(x)
		KeyOutput = self.KeyMatrix(x)
		ValueOutput = self.ValueMatrix(x)

		# 8 Heads
		QueryOutput = QueryOutput.view(BatchSize,SeqLen,NumHeads,HeadDims).transpose(1,2) # B,NumHeads,256,HeadDims
		KeyOutput = KeyOutput.view(BatchSize,SeqLen,NumHeads,HeadDims).transpose(1,2) # B,NumHeads,256,HeadDims
		ValueOutput = ValueOutput.view(BatchSize,SeqLen,NumHeads,HeadDims).transpose(1,2) # B,NumHeads,256,HeadDims

		#Attention(Q, K, V) = (softmax((QK^T)/sqrt(dk)))*V, dk = HeadDims
		KeyOutput = KeyOutput.transpose(2,3) #B,NumHeads,HeadDims,SeqLen
		MatMulOutput = QueryOutput@KeyOutput #B,NumHeads,SeqLen,SeqLen
		MatMulOutput /= math.sqrt(HeadDims)
		MatMulOutput = self.Softmax(MatMulOutput)
		AttentionOutput = MatMulOutput@ValueOutput #B,NumHeads,SeqLen,HeadDims

		AttentionOutput = AttentionOutput.transpose(1,2).contiguous().view(BatchSize,SeqLen,EmbedDim)

		FinalOutput = self.OutputMatrix(AttentionOutput)

		return FinalOutput

In [11]:
class PrismBlock(nn.Module):
	def __init__(self):
		super().__init__()

		self.PseudoQuery = nn.Parameter(torch.randn(EmbedDim)*0.02)
		self.Attention = Attention()
		self.MultiLayerPerceptron = nn.Sequential(
			nn.Linear(in_features=EmbedDim,out_features=EmbedDim*4),
			nn.GELU(),
			nn.Linear(in_features=EmbedDim*4,out_features=EmbedDim)
		)
		self.LayerNorm1 = nn.LayerNorm(EmbedDim)
		self.LayerNorm2 = nn.LayerNorm(EmbedDim)
		self.LayerNorm3 = nn.LayerNorm(EmbedDim)
		self.Softmax = nn.Softmax(dim=0)

	def forward(self,x,HistoryList):
		HistoryList = torch.stack(HistoryList,dim=0) # D,B,SeqLen,EmbedDim
		HistoryListNorm = self.LayerNorm1(HistoryList)

		Scores = (HistoryListNorm*self.PseudoQuery).sum(dim=-1)/math.sqrt(EmbedDim) # D,B,SeqLen
		Weights = self.Softmax(Scores).unsqueeze(-1)

		XRouted = (HistoryList*Weights).sum(dim=0)

		XRoutedNorm = self.LayerNorm2(XRouted)
		x = x + self.Attention(XRoutedNorm)

		xNorm = self.LayerNorm3(x)
		x = x + self.MultiLayerPerceptron(xNorm)

		return x

In [12]:
class PrismModel(nn.Module):
	def __init__(self):
		super().__init__()
		self.HighResConv = nn.Sequential(
			nn.Conv2d(in_channels=1,out_channels=64,kernel_size=3,stride=1,padding=1),
			nn.GELU()
		)
		self.InputLayer = nn.Conv2d(in_channels=64,out_channels=256,kernel_size=16,stride=16) #B,256,16,16
		self.Flatten = nn.Flatten(start_dim=2) #B,256,256
		#Transpose to B,256,256
		self.LearnParam = nn.Parameter(torch.randn((1,256,256)) * 0.02)
		self.ModuleList = nn.ModuleList(
			[PrismBlock() for _ in range(8)]
		)
		self.FinalNorm = nn.LayerNorm(EmbedDim)
		self.Decoder1 = nn.Sequential(
			# 16x16 -> 32x32
			nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
			nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
			nn.GroupNorm(32,256),
			nn.GELU(),
			# 32x32 -> 64x64
			nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
			nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
			nn.GroupNorm(32,256),
			nn.GELU(),
			# 64x64 -> 128x128
			nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
			nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
			nn.GroupNorm(32,256),
			nn.GELU()
		)
		self.Decoder2 = nn.Sequential(
			# 128x128 -> 256x256
			nn.Conv2d(in_channels=320, out_channels=128, kernel_size=3, stride=1, padding=1),
			nn.GroupNorm(16,128),
			nn.GELU(),
			# Final map
			nn.Conv2d(in_channels=128, out_channels=313, kernel_size=3, stride=1, padding=1)
		)

	def forward(self,x):
		SkipFeatures = self.HighResConv(x) # B,64,256,256
		x = self.InputLayer(SkipFeatures) # B,256,16,16
		x = self.Flatten(x) # B,256,256
		x = x.transpose(1,2)
		x = x + self.LearnParam # B,256,256

		HistoryList = [x]
		for module in self.ModuleList:
			x = module(x,HistoryList)
			HistoryList.append(x)

		x = self.FinalNorm(x)

		x = x.transpose(1,2).reshape(-1,256,16,16)

		x = self.Decoder1(x)

		x = nn.functional.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)

		x = torch.cat([x,SkipFeatures], dim=1) # B,320,256,256

		x = self.Decoder2(x)

		return x

In [13]:
def CleanCheckpointStateDict(RawStateDict):
	CleanStateDict = {}
	for key, value in RawStateDict.items():
		CleanKey = key.replace('_orig_mod.', '').replace('module.', '')
		CleanStateDict[CleanKey] = value
	return CleanStateDict

def ColorizeSingleImage(ImagePath, model, BucketsTensor, device):
	ImgBGR = cv2.imread(str(ImagePath))
	if ImgBGR is None:
		print(f"Skipping {ImagePath}, OpenCV could not read it.")
		return None, None
	
	OrigH, OrigW = ImgBGR.shape[:2]
	ImgLAB = cv2.cvtColor(ImgBGR, cv2.COLOR_BGR2LAB)
	LOrig, _, _ = cv2.split(ImgLAB)

	LResized = cv2.resize(LOrig, (256, 256), interpolation=cv2.INTER_CUBIC)
	InputTensor = (torch.from_numpy(LResized).float() / 255.0).unsqueeze(0).unsqueeze(0).to(device)

	with torch.no_grad():
		with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu', dtype=torch.float16):
			Logits = model(InputTensor)
		
		Logits = Logits.float()
		
		Probs = torch.nn.functional.softmax(Logits / Temperature, dim=1)
		PredAB = torch.einsum('b c h w, c d -> b d h w', Probs, BucketsTensor)

	PredABNumpy = PredAB.squeeze(0).cpu().numpy().transpose(1, 2, 0)
	PredABResized = cv2.resize(PredABNumpy, (OrigW, OrigH), interpolation=cv2.INTER_CUBIC)

	PredA = np.clip(PredABResized[..., 0] + 128.0, 0, 255).astype(np.uint8)
	PredB = np.clip(PredABResized[..., 1] + 128.0, 0, 255).astype(np.uint8)

	MergedLAB = cv2.merge([LOrig, PredA, PredB])
	OutputBGR = cv2.cvtColor(MergedLAB, cv2.COLOR_LAB2BGR)

	return ImgBGR, OutputBGR

def ProcessDirectory():
	device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
	print(f"Initializing inference on {device}")

	model = PrismModel().to(device)
	Checkpoint = torch.load(ModelPath, map_location=device, weights_only=False)
	StateDict = Checkpoint.get('ModelState', Checkpoint)
	model.load_state_dict(CleanCheckpointStateDict(StateDict))
	model.eval()

	Buckets = np.load(BucketsPath)
	BucketsTensor = torch.from_numpy(Buckets).float().to(device)

	InPath = Path(InputDir)
	OutPath = Path(OutputDir)
	OutPath.mkdir(parents=True, exist_ok=True)
	
	InputPaths = [p for p in InPath.glob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
	if not InputPaths:
		print(f"No valid images found in {InputDir}")
		return

	print(f"Found {len(InputPaths)} images.")

	for idx, ImgPath in enumerate(InputPaths, 1):
		print(f"[{idx}/{len(InputPaths)}] Colorizing {ImgPath.name}")
		OrigBGR, OutputBGR = ColorizeSingleImage(ImgPath, model, BucketsTensor, device)
		
		if OrigBGR is None:
			continue
		
		OrigRGB = cv2.cvtColor(OrigBGR, cv2.COLOR_BGR2RGB)
		OutputRGB = cv2.cvtColor(OutputBGR, cv2.COLOR_BGR2RGB)
			
		fig, axes = plt.subplots(1, 2, figsize=(12, 6))
		axes[0].imshow(OrigRGB)
		axes[0].set_title("Original Input")
		axes[0].axis("off")
		
		axes[1].imshow(OutputRGB)
		axes[1].set_title("Prism Colorized")
		axes[1].axis("off")
		
		PlotPath = OutPath / f"comparison_{ImgPath.name}"
		plt.savefig(str(PlotPath), bbox_inches='tight')
		plt.close(fig)

In [14]:
if __name__ == '__main__':
	ProcessDirectory()

Initializing inference on cuda
Found 100 images.
[1/100] Colorizing flowers_019922.jpg
[2/100] Colorizing night_049128.jpg
[3/100] Colorizing food_027311.jpg
[4/100] Colorizing buildings_013670.jpg
[5/100] Colorizing indoor_047460.jpg
[6/100] Colorizing animals_009411.jpg
[7/100] Colorizing buildings_013669.jpg
[8/100] Colorizing animals_009409.jpg
[9/100] Colorizing ocean_053207.jpg
[10/100] Colorizing vehicles_064258.jpg
[11/100] Colorizing night_049122.jpg
[12/100] Colorizing ocean_053210.jpg
[13/100] Colorizing flowers_019929.jpg
[14/100] Colorizing flowers_019923.jpg
[15/100] Colorizing vehicles_064261.jpg
[16/100] Colorizing vehicles_064257.jpg
[17/100] Colorizing night_049124.jpg
[18/100] Colorizing scenery_057566.jpg
[19/100] Colorizing flowers_019927.jpg
[20/100] Colorizing night_049130.jpg
[21/100] Colorizing indoor_047454.jpg
[22/100] Colorizing food_027313.jpg
[23/100] Colorizing scenery_057574.jpg
[24/100] Colorizing scenery_057565.jpg
[25/100] Colorizing vehicles_064259.j